In [ ]:
!pip install --user pyspark
!pip install boto3 pandas python-dotenv

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import boto3
import pandas as pd
import os
from dotenv import load_dotenv

# Cria a sessão do Spark
spark = SparkSession.builder.getOrCreate()

In [ ]:
path = './tables/amostragem.csv'
format = 'csv'

In [ ]:
# Criando uma tabela Dataframe da tabela amostragem, colocar inferSchema=True?(determinar se tudo é string ou não?)
table_amostragem_dataframe = spark.read.csv(path, header=True, sep=';')
table_amostragem_dataframe.show()


In [ ]:
table_amostragem_dataframe.printSchema()

In [ ]:
table_amostragem_dataframe = table_amostragem_dataframe.withColumnRenamed('Nome da Tarefa', 'name').withColumnRenamed('Tipo da tarefa ID', 'type_id').withColumnRenamed('Data de Criação', 'date').withColumnRenamed('Status', 'statusId').withColumnRenamed('ID do Usuário', 'user_id').withColumnRenamed('Status Descrição', 'status')
table_amostragem_dataframe.show()

In [ ]:
# Filtrando apenas o Usuário com ID do Jeferson
user = table_amostragem_dataframe.filter((table_amostragem_dataframe['user_id'] == 'b4853fc1f03a3a4cec530a98a94d89ad')& (table_amostragem_dataframe['statusId'] != 3))
user.show(10)

In [ ]:
user = user.withColumn('status', regexp_replace('status',  'Concluído', 'done'))
user = user.withColumn('status', regexp_replace('status', 'A Fazer', 'todo' ))
user = user.withColumn('user_id', regexp_replace('user_id', 'b4853fc1f03a3a4cec530a98a94d89ad', '539c7a3a-d091-7074-17b5-994dde9fccd1' ))

user.show()

In [ ]:
final_table_user = user.drop('Data de Conclusão').drop('Usuário').drop('Tipo da Tarefa').drop('statusId')
final_table_user.show()

In [ ]:
# 1. Carrega as variáveis do arquivo .env
load_dotenv('.env')  # Ou apenas load_dotenv() se o arquivo estiver na raiz

# 2. Acessa as credenciais
aws_access = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')
region = os.getenv('AWS_DEFAULT_REGION')

# 3. Configura o cliente DynamoDB
dynamodb_console = boto3.resource(
    'dynamodb',
    aws_access_key_id=aws_access,
    aws_secret_access_key=aws_secret,
    region_name=region
)

# Lista todas as tabelas do DynamoDB na sua região/conta
for table in dynamodb_console.tables.all():
    print(table.name) # Listando tabelas da conta AWS

In [ ]:

# dynamodb_table = dynamodb_console.Table('pyspark-table')
# print(table_user.key_schema)
type(final_table_user)
print("Total no DataFrame:", final_table_user.count())



In [ ]:
# Use um nome diferente para a tabela do DynamoDB
dynamodb_table = dynamodb_console.Table('pyspark-table')

# Agora sim: usa o DataFrame original
rows = final_table_user.collect()  # Isso vai funcionar agora

with dynamodb_table.batch_writer() as batch:
    for row in rows:
        data = row.asDict()
        batch.put_item(
            Item={
                'PK': f"USERID#{data['user_id']}",
                'SK': f"DATA#{data['date']}",
                'userId': data['user_id'],
                'type_id': (data['type_id']),
                'status': data['status'],
                'name': data['name'],
                'data': data['date']
            }
        )
